<a id="inizio"></a>
# 11 · Capstone: il carico del Nord e la temperatura

⏱ ~110 min &nbsp;·&nbsp; Dati: `../Dati/load_total_north_hourly_2024.xlsx`, `../Dati/load_total_north_hourly_2025.xlsx`, `../Dati/fallback/meteo_milano_2024_2025.json`

**In questo notebook impariamo a**

- caricare, unire e pulire una serie temporale reale, con i salti dell'ora legale
- costruire la serie giornaliera e i profili medi per ora e per giorno della settimana
- unire il carico con la temperatura di Open-Meteo e leggere la relazione tra i due

I dati utilizzati provengono dal [Download Center di Terna](https://dati.terna.it/en/download-center); in
particolare, useremo i dati di **carico elettrico della zona Nord** relativi agli anni 2024 e 2025. Nei
dataset Terna il carico è la potenza richiesta al sistema elettrico in ciascun intervallo temporale, una
grandezza centrale per la pianificazione e la gestione della rete di trasmissione.

Accanto al carico integriamo la **temperatura** di Milano, ottenuta tramite API da
[Open-Meteo](https://open-meteo.com). L'obiettivo è descrivere come il carico del Nord dipende dall'ora
del giorno, dal giorno della settimana e dalla temperatura. Il lavoro è diviso in sei step, ciascuno con
una consegna, una cella da completare e una verifica. Copilot si può usare per i titoli e le etichette
dei grafici, mentre il codice degli step lo scriviamo noi, perché lo scopo del capstone è mettere in
pratica quello che abbiamo visto nei notebook precedenti.

<a id="sez-1"></a>
## 1. Caricamento dei dati e unione

<a id="es-1"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 1 · Caricamento dei dati e unione**

In questo step costruiamo una singola serie storica continua a partire dai dati grezzi.

**Cosa fare.**

1. Leggi i due file Excel presenti nella cartella `../Dati/` in `df_2024` e `df_2025`:
   `load_total_north_hourly_2024.xlsx` e `load_total_north_hourly_2025.xlsx`.
2. Unisci i due dataset **per righe** in `df`, mantenendo lo stesso schema di colonne, in modo da
   ottenere una serie unica che copra l'intero periodo 2024-2025.
3. Controlla la dimensione del dataset, la prima e l'ultima data, il numero medio di righe per giorno,
   da salvare in `righe_giorno`, e i valori mancanti per colonna. Il nome del file contiene `hourly`,
   quindi verifica con il numero di righe per giorno se la frequenza dei dati è effettivamente oraria.

Il risultato è un DataFrame di 70.176 righe e 4 colonne, senza valori mancanti.

</div>

In [ ]:
import pandas as pd

df = ...

righe_giorno = ...
df.isna().sum()

In [ ]:
from corso import verifica
verifica("Step 1")

Il conto dà 96 righe al giorno, quindi i dati sono quartorari, con un valore ogni 15 minuti, anche se il
nome del file dice `hourly`. Ne terremo conto più avanti, quando dalla potenza in MW passeremo
all'energia in MWh, perché ogni valore copre un quarto d'ora.

<a id="sez-2"></a>
## 2. Colonna temporale e prima visualizzazione

<a id="es-2"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 2 · Preparazione della colonna temporale e prima visualizzazione**

In questo step rendiamo la colonna data utilizzabile come variabile temporale e facciamo una prima ispezione visiva della serie.

**Cosa fare.**

1. Controlla con `df.dtypes` che la colonna `Date` sia già di tipo `datetime`; se fosse letta come
   testo, la si converte con `pd.to_datetime`.
2. Ordina il dataset per data, perché ogni file comincia con l'ultimo quarto d'ora dell'anno, e
   reimposta l'indice per avere un ordinamento pulito.
3. Importa Plotly e visualizza **entrambe** le serie nel tempo: `Total Load [MW]` e
   `Forecast Total Load [MW]`.
4. Dopo aver verificato che le due curve siano coerenti, elimina le colonne che non useremo:
   `Forecast Total Load [MW]` e `Bidding Zone`.

Al termine di questo step deve rimanere una serie storica con una colonna temporale in formato
`datetime` e una sola colonna di valori, `Total Load [MW]`.

</div>

In [ ]:
import plotly.express as px

df["Date"] = ...
df = ...

fig = ...
fig.show()

df = ...
df.head()

In [ ]:
from corso import verifica
verifica("Step 2")

<a id="sez-3"></a>
## 3. Ora legale: duplicati e buchi

Prima di aggregare conviene controllare il passo tra un timestamp e il successivo, che in una serie
quartoraria regolare è sempre di 15 minuti. Il passo cambia soltanto nell'ultima domenica di marzo e
nell'ultima domenica di ottobre, i due giorni dell'anno in cui si passa dall'ora solare all'ora legale e
viceversa.

Un primo tentativo consiste nell'assegnare alle date il fuso orario italiano con `tz_localize` e
lasciare a pandas il compito di riconoscere l'ora ripetuta, con l'opzione `ambiguous="infer"`. La
cella qui sotto dà errore apposta, e conviene leggerne l'ultima riga, che spiega il motivo.

In [ ]:
df["Date"].dt.tz_localize("Europe/Rome", ambiguous="infer")

Il messaggio `There are 4 dst switches when there should only be 1` dice che pandas ha trovato
quattro cambi d'ora dove se ne aspettava uno. Succede perché nei file le due copie di ogni quarto
d'ora ripetuto sono intrecciate, e dall'ordine delle righe non si capisce quale appartenga all'ora
legale e quale all'ora solare. Per questa analisi l'ora locale senza fuso è sufficiente, quindi
togliamo i doppioni con `drop_duplicates`.

<a id="es-3"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 3 · Pulizia dei duplicati dell'ora legale**

In questo step lasciamo una riga per timestamp e contiamo i quarti d'ora che mancano.

**Cosa fare.**

1. Calcola in `passi` la differenza tra ogni timestamp e il precedente con `diff()`, conta con
   `value_counts()` quante volte compare ogni valore e chiediti quale sia il passo normale e a che
   cosa corrispondano gli altri due valori.
2. Metti in `duplicati` tutte le copie delle righe con un timestamp ripetuto, usando
   `duplicated(keep=False)`, e guarda in quali giorni e a quali ore cadono.
3. Ordina per `Date`, togli i duplicati con `drop_duplicates(subset="Date")` e reimposta l'indice.
4. Metti in `buchi` le righe che arrivano dopo un salto più lungo di 15 minuti e guarda in quali
   giorni cadono.

</div>

In [ ]:
passi = ...

duplicati = ...

df = ...

buchi = ...
buchi

In [ ]:
from corso import verifica
verifica("Step 3")

<a id="sez-4"></a>
## 4. Aggregazioni temporali e profili medi

Useremo `Date` come colonna temporale e `Total Load [MW]` come variabile di interesse.

<a id="es-4"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 4 · Aggregazioni temporali e profili medi**

In questo step analizziamo la serie quartoraria cambiando punto di vista sul tempo. Costruiamo una
serie giornaliera e osserviamo i pattern medi legati all'ora del giorno e al giorno della settimana.

**Cosa fare.**

**Serie giornaliera (somma)**

1. Parti dal DataFrame quartorario `df`.
2. Imposta la colonna `Date` come indice temporale in `df_indicizzato`.
3. Aggrega per giorno usando `resample("D")` e calcola la **somma** del carico.
4. Riporta `Date` come colonna e salva il risultato in `df_giornaliero`. Aggiungi poi la colonna
   `Energia [MWh]`, pari alla somma divisa per 4, perché un quarto d'ora a una potenza di P MW
   corrisponde a P/4 MWh.
5. Visualizza la serie giornaliera con Plotly.

**Profilo medio giornaliero (media per ora del giorno)**

6. Torna alla serie quartoraria `df`, estrai l'**ora del giorno** dalla colonna `Date` e salvala in
   una nuova colonna `hour`.
7. Raggruppa per `hour` e calcola la **media** del carico in `profilo_orario`.
8. Visualizza il profilo medio giornaliero con Plotly.

**Profilo medio settimanale (media per giorno della settimana)**

9. Sempre partendo da `df`, estrai il **giorno della settimana** dalla colonna `Date` e salvalo in
   una nuova colonna `weekday`, con la convenzione `0 = lunedì, …, 6 = domenica`.
10. Raggruppa per `weekday` e calcola la **media** del carico in `profilo_settimanale`.
11. Visualizza il profilo medio settimanale con Plotly.

Osserva infine i tre grafici e annota in quali periodi l'energia giornaliera scende, a che ore il
carico è più alto e di quanto cala nel weekend, perché questi numeri serviranno nello step 6.

</div>

In [ ]:
# serie giornaliera
df_giornaliero = ...

# profilo medio giornaliero
profilo_orario = ...

# profilo medio settimanale
profilo_settimanale = ...

In [ ]:
from corso import verifica
verifica("Step 4")

<a id="sez-5"></a>
## 5. Recupero e integrazione della temperatura

Al carico aggiungiamo la **temperatura a 2 metri** ricostruita per Milano. Si tratta di una
semplificazione, perché usiamo una sola città come riferimento per tutto il Nord Italia. I dati vengono
dall'API [Historical Weather](https://open-meteo.com/en/docs/historical-weather-api) di Open-Meteo, che
non richiede alcuna chiave e restituisce i valori orari per l'intervallo di date indicato.

<a id="es-5"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 5 · Recupero e integrazione della temperatura**

In questo step integriamo il dato di temperatura con il dato di carico. Per prima cosa va risolta la
differente granularità, perché la temperatura è oraria mentre il carico è quartorario; la temperatura va
quindi portata da un valore ogni ora a un valore ogni 15 minuti.

**Cosa fare.**

1. Chiedi all'archivio storico di Open-Meteo la temperatura oraria di Milano, alla latitudine
   45.4642 e longitudine 9.19, dal primo all'ultimo giorno del carico. L'indirizzo è
   `https://archive-api.open-meteo.com/v1/archive` e i parametri sono `latitude`, `longitude`,
   `start_date` ed `end_date`, con le date come testo `AAAA-MM-GG`, più `hourly="temperature_2m"` e
   `timezone="Europe/Rome"`. Salva in `orario` la chiave `hourly` della risposta.
2. Nella cella che segue quella dei dati di riserva, trasforma `orario` in `meteo` con
   `pd.DataFrame`.
3. Converti `time` in datetime, rinomina le colonne in `Date` e `Temperature_C`, imposta `Date` come
   indice e ordinalo con `sort_index()`.
4. Porta la temperatura a frequenza quartoraria con `resample("15min")` seguito da
   `interpolate(method="linear")`, così da poterla unire al carico, poi riporta `Date` come colonna e
   salva il risultato in `meteo_15`.
5. Fai il merge con il DataFrame del carico `df`, tenendo tutte le sue righe, e salva il risultato in
   `df_completo`.
6. Dopo il merge gli ultimi quarti d'ora del 2025 restano senza temperatura; riempili con `ffill()`,
   che ripete l'ultimo valore disponibile.
7. Visualizza la temperatura nel tempo con `px.line()`.

</div>

In [ ]:
import requests

data_inizio = ...
data_fine = ...
url = "https://archive-api.open-meteo.com/v1/archive"
parametri = ...

orario = ...

Se la rete non risponde, la cella qui sotto carica in `orario` le temperature di esempio della
cartella `../Dati/fallback/`, che non sono misurate; se l'API ha risposto, la cella va saltata.

In [ ]:
import json

with open("../Dati/fallback/meteo_milano_2024_2025.json", encoding="utf-8") as f:
    orario = json.load(f)["hourly"]
len(orario["time"])

In [ ]:
# punti 2-7: da orario a df_completo
meteo = ...

meteo_15 = ...

df_completo = ...

In [ ]:
from corso import verifica
verifica("Step 5")

<a id="sez-6"></a>
## 6. Carico e temperatura

<a id="es-6"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Step 6 · Carico e temperatura**

In questo step costruiamo uno scatter con un punto per giorno, la temperatura media sulle x, il carico medio sulle y e un colore diverso per ogni mese. Dalla forma della nuvola di punti si legge come il carico dipende dalla temperatura.

**Cosa fare.**

1. Calcola in `giornaliero` le medie giornaliere di `df_completo`, impostando `Date` come indice,
   selezionando le colonne `Total Load [MW]` e `Temperature_C`, applicando `resample("D").mean()` e
   riportando infine `Date` come colonna con `reset_index()`.
2. Aggiungi la colonna `mese` con il nome del mese, ricavato con `dt.month_name()`, in modo che lo
   scatter abbia un colore per ogni mese invece di una scala continua.
3. Disegna lo scatter con `px.scatter`, con la temperatura sulle x, il carico sulle y e
   `color="mese"`, e osserva che forma ha la nuvola e a quale temperatura il carico è più basso.
4. Completa le cinque frasi della cella che segue la verifica, usando i numeri e i grafici ottenuti
   negli step precedenti.

</div>

In [ ]:
giornaliero = ...

fig = ...
fig.show()

In [ ]:
from corso import verifica
verifica("Step 6")

**Il carico del Nord, 2024-2025**

1. I dati sono ...: ... righe pulite, ... doppioni di ottobre tolti, ... quarti d'ora di marzo mancanti.
2. L'energia giornaliera va da circa ... a circa ... MWh; i giorni più leggeri sono ...
3. Nella giornata media il carico è minimo alle ... e massimo alle ...
4. La domenica il carico medio sta intorno a ... MW, un giorno feriale intorno a ... MW.
5. Il carico è più basso intorno a ... °C e sale con il freddo e con il caldo; pesa di più ...

Fonte della temperatura: ... (Open-Meteo oppure i dati di esempio di `../Dati/fallback/`).

---

**Fine del notebook 11.** Precedente: [10 · Agenti per il coding](10_Agenti_per_il_coding.ipynb)

<sub>Blocco 4 · Agenti per il coding e caso d'uso end-to-end · Giornata 2 · Aula Avanzata</sub>